# neurolib: Whole-Brain Network Modeling

Simulate whole-brain dynamics using the ALN model connected via empirical DTI structural connectivity. Compare simulated BOLD to resting-state fMRI data.

## Prerequisites
- Understanding of neural mass/mean-field models and whole-brain dynamics
- Familiarity with structural connectivity (DTI/DSI) and functional connectivity (fMRI)
- Knowledge of the BOLD signal and hemodynamic response
- Basic Python, NumPy, and matplotlib skills

## Setup
Run the cell below to install dependencies and download data.

In [ ]:
!pip install neurolib nilearn matplotlib numpy

In [ ]:
import neurolib
from neurolib.models.aln import ALNModel
from neurolib.utils.loadData import Dataset
from neurolib.utils import brainplot
import matplotlib.pyplot as plt
import numpy as np

# Load empirical SC/FC data (80-region parcellation, DSI studio)
ds = Dataset('hcp')
print(f'Structural connectivity: {ds.Cmat.shape}')
print(f'Delay matrix: {ds.Dmat.shape}')

# Build whole-brain ALN model
model = ALNModel(Cmat=ds.Cmat, Dmat=ds.Dmat)
model.params['duration'] = 60 * 1000  # 60 seconds
model.params['dt'] = 0.1              # 0.1 ms resolution
model.params['bold'] = True           # compute Balloon-Windkessel BOLD signal

print('Running whole-brain simulation...')
model.run()
print('Done. Output shape (exc rate):', model.rates_exc.shape)

In [ ]:
# Plot simulated excitatory firing rates across all 80 brain regions
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

t = np.arange(model.rates_exc.shape[1]) * model.params['dt']
im = axes[0].imshow(model.rates_exc, aspect='auto', cmap='hot',
                    extent=[t[0], t[-1], 0, model.rates_exc.shape[0]])
axes[0].set_xlabel('Time (ms)'); axes[0].set_ylabel('Brain area')
axes[0].set_title('Excitatory firing rates \u2014 80 brain areas (whole-brain ALN model)')
plt.colorbar(im, ax=axes[0], label='Rate (Hz)')

mean_rates = model.rates_exc.mean(axis=1)
axes[1].bar(range(len(mean_rates)), mean_rates, color='steelblue')
axes[1].set_xlabel('Brain area index'); axes[1].set_ylabel('Mean firing rate (Hz)')
axes[1].set_title('Mean excitatory rate per area')
plt.tight_layout(); plt.show()

# Simulated BOLD signal (Balloon-Windkessel hemodynamic model)
bold = model.BOLD.BOLD  # shape: (areas, time_bold)
print(f'Simulated BOLD shape: {bold.shape}  (areas \u00d7 TR-sampled timepoints)')

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(bold, aspect='auto', cmap='RdBu_r')
ax.set_xlabel('TR'); ax.set_ylabel('Brain area')
ax.set_title('Simulated BOLD signal \u2014 whole-brain ALN model')
plt.colorbar(im, ax=ax, label='BOLD (a.u.)')
plt.tight_layout(); plt.show()

# Simulated functional connectivity vs empirical FC
from nilearn.connectome import ConnectivityMeasure
conn = ConnectivityMeasure(kind='correlation')
sim_fc = conn.fit_transform([bold.T])[0]

# ds.FCs is a list of empirical FC matrices from HCP subjects (ds.FC does not exist)
empirical_fc = ds.FCs[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(sim_fc, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title('Simulated FC (ALN model)'); axes[0].set_xlabel('Area'); axes[0].set_ylabel('Area')
axes[1].imshow(empirical_fc, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_title('Empirical FC (HCP resting-state)'); axes[1].set_xlabel('Area')
plt.suptitle('Simulated vs Empirical Functional Connectivity')
plt.tight_layout(); plt.show()

## References

- Cakan, C., & Obermayer, K. (2020). Biophysically grounded mean-field models of neural populations under electrical stimulation. *PLOS Computational Biology*, 16(4), e1007822. https://doi.org/10.1371/journal.pcbi.1007822
- Cakan, C., Jajcay, N., & Obermayer, K. (2021). neurolib: A simulation framework for whole-brain neural mass modeling. *Cognitive Computation*, 15, 1132\u20131152. https://doi.org/10.1007/s12559-021-09931-9
- Deco, G., Jirsa, V. K., Robinson, P. A., Breakspear, M., & Friston, K. (2008). The dynamic brain: from spiking neurons to neural masses and cortical fields. *PLOS Computational Biology*, 4(8), e1000092.
- Van Essen, D. C., Smith, S. M., Barch, D. M., Behrens, T. E. J., Yacoub, E., & Ugurbil, K. (2013). The WU-Minn Human Connectome Project: an overview. *NeuroImage*, 80, 62\u201379.